In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Helper Function for Text Cleaning:

Implement a Helper Function as per Text Preprocessing Notebook and Complete the following pipeline.

# Build a Text Cleaning Pipeline

In [4]:
import re
import pandas as pd
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
custom_stopwords = ['rt', '@']
stop_words.update(custom_stopwords)

def lower_order(text):
    return text.lower()

def remove_urls(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)

def remove_emoji(text):
    return re.sub(r'[^\x00-\x7F]+', ' ', text)

def removeunwanted_characters(text):
    text = re.sub(r"@[A-Za-z0-9_]+", " ", text)
    text = re.sub(r"#[A-Za-z0-9_]+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

#  FINAL PIPELINE
def text_cleaning_pipeline(dataset, rule="lemmatize"):

    if pd.isna(dataset):
        return ""

    # Lowercase
    data = lower_order(dataset)

    # Remove URLs
    data = remove_urls(data)

    # Remove emojis
    data = remove_emoji(data)

    # Remove unwanted characters
    data = removeunwanted_characters(data)

    # Tokenization
    tokens = data.split()

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatize / Stem
    wordnet = WordNetLemmatizer()
    porter = PorterStemmer()

    if rule == "lemmatize":
        tokens = [wordnet.lemmatize(word, pos='v') for word in tokens]
    elif rule == "stem":
        tokens = [porter.stem(word) for word in tokens]
    else:
        print("Pick between lemmatize or stem")

    return " ".join(tokens)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


### Load Dataset

In [5]:
import pandas as pd

data = pd.read_csv("/content/drive/MyDrive/AI_ML/Week8/trum_tweet_sentiment_analysis.csv", encoding="ISO-8859-1")

data.head()

,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


### Apply Cleaning

In [6]:
data["cleaned_text"] = data["text"].apply(lambda x: text_cleaning_pipeline(x))

data[["text", "cleaned_text"]].head()

,text,cleaned_text
0,RT @JohnLeguizamo: #trump not draining swamp b...,drain swamp taxpayer dollars trip advertise pr...
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,icymi hackers rig fm radio station play anti t...
2,Trump protests: LGBTQ rally in New York https:...,trump protest lgbtq rally new york via
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",hi piers morgan david beckham awful donald tru...
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,tech firm sue buzzfeed publish unverified trum...


### Train-Test Split

In [10]:
print(data.columns)

Index(['text', 'Sentiment', 'cleaned_text'], dtype='object')


In [11]:
from sklearn.model_selection import train_test_split

X = data["cleaned_text"]
y = data["Sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### TF-IDF Vectorization

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

### Train Model (Logistic Regression)

In [13]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

LogisticRegression()

### Evaluate Model

In [14]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test_tfidf)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.96      0.95    248563
           1       0.92      0.88      0.90    121462

    accuracy                           0.94    370025
   macro avg       0.93      0.92      0.93    370025
weighted avg       0.93      0.94      0.94    370025



# Text Classification using Machine Learning Models


### 📝 Instructions: Trump Tweet Sentiment Classification

1. **Load the Dataset**  
   Load the dataset named `"trump_tweet_sentiment_analysis.csv"` using `pandas`. Ensure the dataset contains at least two columns: `"text"` and `"label"`.

2. **Text Cleaning and Tokenization**  
   Apply a text preprocessing pipeline to the `"text"` column. This should include:
   - Lowercasing the text  
   - Removing URLs, mentions, punctuation, and special characters  
   - Removing stopwords  
   - Tokenization (optional: stemming or lemmatization)
   - "Complete the above function"

3. **Train-Test Split**  
   Split the cleaned and tokenized dataset into **training** and **testing** sets using `train_test_split` from `sklearn.model_selection`.

4. **TF-IDF Vectorization**  
   Import and use the `TfidfVectorizer` from `sklearn.feature_extraction.text` to transform the training and testing texts into numerical feature vectors.

5. **Model Training and Evaluation**  
   Import **Logistic Regression** (or any machine learning model of your choice) from `sklearn.linear_model`. Train it on the TF-IDF-embedded training data, then evaluate it using the test set.  
   - Print the **classification report** using `classification_report` from `sklearn.metrics`.
